# Car Price Prediction

**Task:** Predict the price of a car listing based on its characteristics.

**Type:** Regression (supervised learning).

**y:** `price` — the asking price in the listing (not the final transaction price).

**X:** Vehicle characteristics — `brand`, `model`, `year`, `mileage`, `engine`, `fuel`, `gearbox`, `horsepower`, etc.

**Observation:** One car sale listing.

**Metrics:** MAE (primary), RMSE, R². The use of MAPE and/or a log-transformed target will be decided after EDA.

**Success criterion:** Significant improvement over the baseline (median prediction) + an understanding of where and why the model makes errors.

**Known limitations:** The asking price is not the same as the final transaction price. Part of the price variation is driven by seller behavior and is therefore irreducible from the available vehicle characteristics.


In [24]:
import pandas as pd
import numpy as np

In [25]:
df = pd.read_csv('data/used_cars.csv')

print('DF shape(rows, columns) ', df.shape)
df.head()

DF shape(rows, columns)  (4009, 12)


,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [26]:
df.sample(10, random_state=42)

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
2580,Lexus,IS 300 Base,2018,"50,992 mi.",Gasoline,260.0HP 3.5L V6 Cylinder Engine Gasoline Fuel,A/T,White,Gray,At least 1 accident or damage reported,Yes,"$28,000"
3660,Chevrolet,Impala Base,2004,"64,500 mi.",Gasoline,180.0HP 3.4L V6 Cylinder Engine Gasoline Fuel,A/T,Beige,Beige,None reported,Yes,"$5,900"
897,RAM,2500 SLT,2017,"86,000 mi.",Diesel,350.0HP 6.7L Straight 6 Cylinder Engine Diesel...,6-Speed A/T,Gray,Gray,At least 1 accident or damage reported,Yes,"$41,000"
2091,Mercedes-Benz,SL-Class SL 550,2013,"24,933 mi.",Gasoline,429.0HP 4.6L 8 Cylinder Engine Gasoline Fuel,Transmission w/Dual Shift Mode,Silver,Red,At least 1 accident or damage reported,Yes,"$40,250"
1044,Ford,Shelby GT350R Base,2018,"18,500 mi.",Gasoline,526.0HP 5.2L 8 Cylinder Engine Gasoline Fuel,M/T,Blue,Black,At least 1 accident or damage reported,Yes,"$77,999"
2320,GMC,Yukon SLT,2018,"85,500 mi.",Gasoline,355.0HP 5.3L 8 Cylinder Engine Gasoline Fuel,A/T,Black,Black,At least 1 accident or damage reported,Yes,"$35,899"
465,Volvo,V60 Cross Country T5,2020,"10,500 mi.",Gasoline,250.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,8-Speed A/T,White,Black,None reported,Yes,"$36,000"
196,Nissan,Titan XD SV,2020,"10,001 mi.",Gasoline,5.6L V8 32V GDI DOHC,9-Speed Automatic,Black,Black,None reported,Yes,"$47,214"
3113,BMW,330 i,2005,"59,300 mi.",Gasoline,225.0HP 3.0L Straight 6 Cylinder Engine Gasoli...,M/T,Red,Black,None reported,Yes,"$30,900"
3553,Lexus,RX 330 Base,2006,"110,250 mi.",Gasoline,230.0HP 3.3L V6 Cylinder Engine Gasoline Fuel,A/T,White,Beige,None reported,Yes,"$11,000"


In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   brand         4009 non-null   str  
 1   model         4009 non-null   str  
 2   model_year    4009 non-null   int64
 3   milage        4009 non-null   str  
 4   fuel_type     3839 non-null   str  
 5   engine        4009 non-null   str  
 6   transmission  4009 non-null   str  
 7   ext_col       4009 non-null   str  
 8   int_col       4009 non-null   str  
 9   accident      3896 non-null   str  
 10  clean_title   3413 non-null   str  
 11  price         4009 non-null   str  
dtypes: int64(1), str(11)
memory usage: 892.6 KB


In [28]:
missing = pd.DataFrame({
    'isna' : df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(2),
})
missing.sort_values('isna',ascending=False)

,isna,pct_missing
clean_title,596,14.87
fuel_type,170,4.24
accident,113,2.82
brand,0,0.00
milage,0,0.00
model_year,0,0.00
model,0,0.00
engine,0,0.00
ext_col,0,0.00
transmission,0,0.00


In [29]:
unique_values = pd.DataFrame(
    {
        'n_unique': df.nunique(),
        'dtypes' : df.dtypes,
    }
).sort_values('n_unique',ascending=False)
unique_values

,n_unique,dtypes
milage,2818,str
model,1898,str
price,1569,str
engine,1146,str
ext_col,319,str
int_col,156,str
transmission,62,str
brand,57,str
model_year,34,int64
fuel_type,7,str


In [30]:
df.describe()

,model_year
count,4009.000000
mean,2015.515590
std,6.104816
min,1974.000000
25%,2012.000000
50%,2017.000000
75%,2020.000000
max,2024.000000


In [31]:
print('Duplicate: ',df.duplicated().sum())
print('Duplicate without price: ',df.drop(columns=['price']).duplicated().sum())

Duplicate:  0
Duplicate without price:  0


In [34]:
SUSPECTS = {'-', '–','—','NA','n/a','N/A','na','unk','unknown','Unknown','None','none','',' ','?',',', 'null','NULL'}
for col in df.columns:
    if df[col].dtype == 'int64':
        continue
    hits = df[col][df[col].isin(SUSPECTS)]
    if len(hits):
        print(f"{col:14s} -> {hits.value_counts().to_dict()}")

fuel_type      -> {'–': 45}
engine         -> {'–': 45}
transmission   -> {'–': 4}
ext_col        -> {'–': 15}
int_col        -> {'–': 133}


In [39]:
for col in ["fuel_type", "accident", "clean_title", "brand"]:
    print(f"\n===== {col} (n_unique={df[col].nunique()}) =====")
    print(df[col].value_counts(dropna=False))


===== fuel_type (n_unique=7) =====
fuel_type
Gasoline          3309
Hybrid             194
NaN                170
E85 Flex Fuel      139
Diesel             116
–                   45
Plug-In Hybrid      34
not supported        2
Name: count, dtype: int64

===== accident (n_unique=2) =====
accident
None reported                             2910
At least 1 accident or damage reported     986
NaN                                        113
Name: count, dtype: int64

===== clean_title (n_unique=1) =====
clean_title
Yes    3413
NaN     596
Name: count, dtype: int64

===== brand (n_unique=57) =====
brand
Ford             386
BMW              375
Mercedes-Benz    315
Chevrolet        292
Porsche          201
Audi             200
Toyota           199
Lexus            163
Jeep             143
Land             130
Nissan           116
Cadillac         107
GMC               91
RAM               91
Dodge             90
Tesla             87
Kia               76
Hyundai           72
Acura           

In [40]:
for col in ["model", "transmission", "ext_col", "engine"]:
    vc = df[col].value_counts()
    print(f"{col:14s} n_unique={len(vc):5d} | "
          f"appear once={(vc == 1).sum():5d} | "
          f"rows covered by top-20={vc.head(20).sum() / len(df):.1%}")

model          n_unique= 1898 | appear once= 1082 | rows covered by top-20=8.0%
transmission   n_unique=   62 | appear once=   22 | rows covered by top-20=96.3%
ext_col        n_unique=  319 | appear once=  214 | rows covered by top-20=88.7%
engine         n_unique= 1146 | appear once=  492 | rows covered by top-20=15.6%


In [41]:
price_num  = pd.to_numeric(df["price"].str.replace(r"[\$,]", "", regex=True),errors="coerce")
milage_num = pd.to_numeric(df["milage"].str.replace(r"[,]|\smi\.", "", regex=True),errors="coerce")

print("price  — failed to parse:", price_num.isna().sum())
print("milage — failed to parse:", milage_num.isna().sum())

pd.DataFrame({"price": price_num, "milage": milage_num}) \
  .describe(percentiles=[.01, .25, .5, .75, .95, .99]).round(0)

price  — failed to parse: 0
milage — failed to parse: 0


,price,milage
count,4009.0,4009.0
mean,44553.0,64718.0
std,78711.0,52297.0
min,2000.0,100.0
1%,4000.0,635.0
25%,17200.0,23044.0
50%,31000.0,52775.0
75%,49990.0,94100.0
95%,111600.0,165000.0
99%,272713.0,222428.0


In [42]:
na_flags = df[["fuel_type", "accident", "clean_title"]].isna()
print("Rows with at least one NaN:", na_flags.any(axis=1).sum())
print("Rows with all three NaN:  ", na_flags.all(axis=1).sum())
na_flags.corr().round(2)

Rows with at least one NaN: 740
Rows with all three NaN:   4


,fuel_type,accident,clean_title
fuel_type,1.00,-0.01,0.00
accident,-0.01,1.00,0.41
clean_title,0.00,0.41,1.00
